# 🧪 L1 & L2 Regularization — A/B Test (TensorFlow)

**What you will learn:**
- Why overfitting happens and what it looks like
- How L1 (Lasso) and L2 (Ridge / Weight Decay) regularization work
- Side-by-side A/B comparison of: No Reg | L1 | L2 | L1+L2 (ElasticNet)
- How to read training curves, weight distributions, and decision boundaries

**Dataset:** Synthetic 2D moons — deliberately small so overfitting is easy to trigger.


In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Imports
# ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print('GPU available:', len(tf.config.list_physical_devices("GPU")) > 0)

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Dataset: Intentionally small to trigger overfitting
# ─────────────────────────────────────────────
N_SAMPLES = 200          # small dataset → easy to overfit
NOISE     = 0.25         # label noise makes it harder
TEST_SIZE = 0.3

X, y = make_moons(n_samples=N_SAMPLES, noise=NOISE, random_state=42)

# Scale features to zero-mean, unit-variance
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')

# ── Quick visualisation ──
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(*X_train[y_train==0].T, c='#4C9BE8', s=25, label='Class 0 (train)', alpha=0.7)
ax.scatter(*X_train[y_train==1].T, c='#E84C4C', s=25, label='Class 1 (train)', alpha=0.7)
ax.scatter(*X_val[y_val==0].T,   c='#4C9BE8', s=25, marker='^', label='Class 0 (val)',   alpha=0.4)
ax.scatter(*X_val[y_val==1].T,   c='#E84C4C', s=25, marker='^', label='Class 1 (val)',   alpha=0.4)
ax.set_title('Two-Moons Dataset (small + noisy)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Model factory
# Deliberately LARGE (capacity >> data size) so it overfits without regularization
# ─────────────────────────────────────────────
def build_model(reg_type='none', lam=0.01, name='model'):
    """
    reg_type : 'none' | 'l1' | 'l2' | 'l1_l2'
    lam      : regularization strength (lambda)
    """
    # Choose regularizer
    if reg_type == 'none':
        reg = None                                         # ← no penalty
    elif reg_type == 'l1':
        reg = regularizers.l1(lam)                         # ← |w| penalty
    elif reg_type == 'l2':
        reg = regularizers.l2(lam)                         # ← w² penalty
    elif reg_type == 'l1_l2':
        reg = regularizers.l1_l2(l1=lam, l2=lam)          # ← ElasticNet
    else:
        raise ValueError(f'Unknown reg_type: {reg_type}')

    # Large MLP — 4 hidden layers × 128 units
    # This is deliberately over-parameterised for our 140-sample training set
    model = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(128, activation='relu', kernel_regularizer=reg, name='dense_1'),
        layers.Dense(128, activation='relu', kernel_regularizer=reg, name='dense_2'),
        layers.Dense(128, activation='relu', kernel_regularizer=reg, name='dense_3'),
        layers.Dense(128, activation='relu', kernel_regularizer=reg, name='dense_4'),
        layers.Dense(1,   activation='sigmoid',                       name='output'),
    ], name=name)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Quick sanity check
build_model('l2').summary()

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — A/B Training loop
# Train all 4 variants, store history for comparison
# ─────────────────────────────────────────────
EPOCHS   = 300
BS       = 32
LAMBDA   = 0.001     # regularization strength — try 0.0001, 0.001, 0.01 to see the effect

VARIANTS = [
    ('none',   'No Regularization',  '#E84C4C'),
    ('l2',     'L2  (λ=%.4f)' % LAMBDA, '#4C9BE8'),
    ('l1',     'L1  (λ=%.4f)' % LAMBDA, '#2ECC71'),
    ('l1_l2',  'L1+L2 ElasticNet',   '#F39C12'),
]

histories = {}
models    = {}

for reg_type, label, color in VARIANTS:
    print(f'\n── Training: {label} ──')
    m = build_model(reg_type=reg_type, lam=LAMBDA, name=reg_type)
    h = m.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BS,
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=30, restore_best_weights=True
        )]
    )
    histories[reg_type] = h.history
    models[reg_type]    = m
    val_acc = max(h.history['val_accuracy'])
    print(f'  Best val accuracy: {val_acc:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Plot A: Training vs Validation Loss curves
# Key insight: gap between train and val = overfitting
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
fig.suptitle('Training vs Validation Loss\n(gap = overfitting)', fontsize=14, fontweight='bold')

for ax, (reg_type, label, color) in zip(axes, VARIANTS):
    h   = histories[reg_type]
    ep  = range(1, len(h['loss']) + 1)
    ax.plot(ep, h['loss'],     color=color,  lw=2,   label='Train loss')
    ax.plot(ep, h['val_loss'], color=color,  lw=2,   ls='--', label='Val loss', alpha=0.8)
    # Shade the overfitting gap
    ax.fill_between(ep, h['loss'], h['val_loss'],
                    alpha=0.15, color=color, label='Gap (overfit)')
    ax.set_title(label, fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Loss')
plt.tight_layout(); plt.show()

print()
print('💡 KEY OBSERVATION:')
print('  • No-reg: large gap (train loss << val loss) → OVERFITTING')
print('  • L2/L1:  smaller gap → regularization is working')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Plot B: Decision boundaries side-by-side
# KEY insight: no-reg = jagged boundary, L2 = smooth, L1 = sharper
# ─────────────────────────────────────────────
def plot_decision_boundary(ax, model, X, y, title, color):
    h = 0.03
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid   = np.c_[xx.ravel(), yy.ravel()]
    Z      = model.predict(grid, verbose=0).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu', levels=20)
    ax.contour( xx, yy, Z, levels=[0.5], colors=[color], linewidths=2)
    ax.scatter(*X[y==0].T, c='#4C9BE8', s=20, edgecolors='k', lw=0.3, label='Class 0')
    ax.scatter(*X[y==1].T, c='#E84C4C', s=20, edgecolors='k', lw=0.3, label='Class 1')
    val_acc = np.mean((model.predict(X_val, verbose=0).ravel() > 0.5).astype(int) == y_val)
    ax.set_title(f'{title}\nVal acc: {val_acc:.3f}', fontsize=10, fontweight='bold', color=color)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Decision Boundaries — A/B Comparison', fontsize=14, fontweight='bold')

X_all = np.vstack([X_train, X_val])
y_all = np.concatenate([y_train, y_val])

for ax, (reg_type, label, color) in zip(axes, VARIANTS):
    plot_decision_boundary(ax, models[reg_type], X_all, y_all, label, color)

plt.tight_layout(); plt.show()

print()
print('💡 KEY OBSERVATION:')
print('  • No-reg: wiggly, jagged boundary fitting noise')
print('  • L2:     smooth curved boundary — generalizes better')
print('  • L1:     can produce more angular / piecewise boundary')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Plot C: Weight distributions
# L1 → sparse (many exact zeros), L2 → small but non-zero
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
fig.suptitle('Weight Distributions After Training\n(L1 → sparse | L2 → small but dense)',
             fontsize=13, fontweight='bold')

for ax, (reg_type, label, color) in zip(axes, VARIANTS):
    all_weights = []
    for layer in models[reg_type].layers:
        for w in layer.get_weights():
            if w.ndim > 1:   # kernel only, skip biases
                all_weights.append(w.ravel())
    all_w = np.concatenate(all_weights)

    ax.hist(all_w, bins=80, color=color, alpha=0.8, density=True)
    ax.axvline(0, color='black', lw=1.5, ls='--')

    pct_near_zero = np.mean(np.abs(all_w) < 0.01) * 100
    ax.set_title(f'{label}\n{pct_near_zero:.1f}% weights ≈ 0', fontsize=10,
                 fontweight='bold', color=color)
    ax.set_xlabel('Weight value')
    ax.grid(alpha=0.3)
    stats = f'μ={all_w.mean():.3f}\nσ={all_w.std():.3f}'
    ax.text(0.98, 0.95, stats, transform=ax.transAxes,
            va='top', ha='right', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout(); plt.show()

print()
print('💡 KEY OBSERVATION:')
print('  • L1 produces SPARSITY — many weights are exactly 0 (feature selection)')
print('  • L2 produces SMALL weights — distributed across all features')
print('  • No-reg: large spread (weights can grow large to fit noise)')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Plot D: Lambda sweep — How regularization strength affects val accuracy
# ─────────────────────────────────────────────
lambdas    = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]
results    = {'l1': [], 'l2': []}

for lam in lambdas:
    for rt in ['l1', 'l2']:
        m = build_model(reg_type=rt, lam=lam)
        m.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=200, batch_size=32, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])
        val_acc = max(m.history.history['val_accuracy'])
        results[rt].append(val_acc)
    print(f'λ={lam:.4f} done')

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(lambdas, results['l2'], 'o-', color='#4C9BE8', lw=2, ms=7, label='L2')
ax.semilogx(lambdas, results['l1'], 's-', color='#2ECC71', lw=2, ms=7, label='L1')
ax.set_xlabel('Lambda (regularization strength) — log scale', fontsize=12)
ax.set_ylabel('Best Validation Accuracy', fontsize=12)
ax.set_title('Lambda Sweep — Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print()
print('💡 KEY OBSERVATION:')
print('  • Too small λ  → undoes regularization → overfitting')
print('  • Too large λ  → weights forced to zero → underfitting')
print('  • Sweet spot   → best generalisation (peak of the curve)')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Final summary table
# ─────────────────────────────────────────────
print('=' * 60)
print(f'{"Variant":<20} {"Train Acc":>10} {"Val Acc":>10} {"Gap":>8}')
print('=' * 60)
for reg_type, label, _ in VARIANTS:
    h = histories[reg_type]
    train_acc = max(h['accuracy'])
    val_acc   = max(h['val_accuracy'])
    gap       = train_acc - val_acc
    flag      = ' ⚠️  OVERFIT' if gap > 0.08 else ' ✅'
    print(f'{label:<20} {train_acc:>10.4f} {val_acc:>10.4f} {gap:>8.4f}{flag}')
print('=' * 60)

print("""
FORMULA RECAP
─────────────────────────────────────────────────
L_total  =  L_task  +  λ · R(W)

L2 (Ridge):  R(W) = ||W||²  = Σ wᵢ²
  → gradient: 2λW   → weights shrink proportionally
  → encourages small, dense weights

L1 (Lasso):  R(W) = ||W||₁  = Σ|wᵢ|
  → gradient: λ·sign(W)   → constant push toward 0
  → encourages EXACT zeros → feature selection

ElasticNet:  R(W) = λ₁||W||₁ + λ₂||W||²
  → combines sparsity (L1) + stability (L2)
""")